In [1]:
import cv2
import torch
import ultralytics
import torchvision
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

In [3]:
from ultralytics import solutions
from ultralytics import YOLO

In [6]:
import os

In [33]:
import shutil

In [42]:
def mask_color_create(frame):
    """
    Обрезаем подаваемый фрейм cv2.imread(путь к изображению)
    и определяем число пикселей нужного цвета в маске
    если число пикселей нужного цвета находится в пределах определенного значения,
    выводим True и после этого продолжаем обработку изображения, инача False - самокат не наш
    """
    # cropp_frame = cropper(frame)

    # словарь: ключ - название цветового канала, значение - число пикселей данного канала на изображении
    colors_pixels = dict()

    # переводим в удобоваримый для cv2 формат - HSV
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Устанавливаем крайиние границы для фиолетового и синего цвета
    lower_purple = np.array([115, 100, 100])
    upper_purple = np.array([150, 255, 255])
    
    lower_blue = np.array([104, 117, 92]) 
    upper_blue = np.array([114, 255, 255])

    # Применяем маски для изображения переконвертированного в hsv 
    mask_purple = cv2.inRange(hsv, lower_purple, upper_purple)
    mask_blue = cv2.inRange(hsv, lower_blue, upper_blue)

    # cv2.imshow('Blue mask', mask_blue)
    # resp = cv2.bitwise_and(frame, frame, mask=mask_blue)
    # cv2.imshow('result', resp)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()

    # Считаем число ненулевых пикселей по каждой маске
    purp_pixels = cv2.countNonZero(mask_purple)
    bl_pixels = cv2.countNonZero(mask_blue)

    # Заполням словарь
    colors_pixels['purple'] = purp_pixels
    colors_pixels['blue'] = bl_pixels

    # True, если число пикселей больше 1000, инчае - False
    flag_blue = True if colors_pixels['blue'] > 2000 else False
    flag_purple = True if colors_pixels['purple'] > 2000 else False

    return flag_blue | flag_purple

In [46]:
def activate(path_to_cropp: str, path_cropped: str = 'cropped-detections/'):
    """
    Функция, которая получит изображение, применит модель YOLO для детекции
    Добавит его в директорию с обрезанными изображениями
    Вызовет функцию для классификации, выведет ее значение
    После - удалит файл 
    """
    # Обертка, которая применит модель детекции и возьмет отдельные рамки для изображений
    cropper = solutions.ObjectCropper(
    model='runs/detect/train/weights/last.pt',
    )
    to_cropp = cropper(cv2.imread(path_to_cropp))
    total = dict()

    cropped_items = []
    # Цикл, в котором обрабатываются изображения из директории path_cropped, куда сохраняются обрезанные изображения
    for i in range(len(os.listdir(path_cropped))):
        cropped_item = os.path.join(path_cropped, os.listdir(path_cropped)[i])
        cropped_items.append(cropped_item)
        frame = cv2.imread(cropped_item)
        data = mask_color_create(frame=frame)
        total[i] = data
    # Удаляем директорию
    shutil.rmtree(path_cropped)
    
    return total

In [47]:
print(activate(path_to_cropp='resized_sams/IMG_2534.jpg'))

Ultralytics Solutions: ✅ {'region': None, 'show_in': True, 'show_out': True, 'colormap': None, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'json_file': None, 'records': 5, 'model': 'runs/detect/train/weights/last.pt'}

0: 640x480 1 samokat, 44.2ms
Speed: 1.7ms preprocess, 44.2ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 480)
🚀 Results: SolutionResults(total_crop_objects=1)
{0: True}
